In [ ]:
import torch  # Imports the PyTorch library for deep learning operations

print("PyTorch version:", torch.__version__)  # Prints the installed PyTorch version
print("CUDA available:", torch.cuda.is_available())  # Checks if CUDA (NVIDIA GPU support) is available

if torch.cuda.is_available():  # If a CUDA-enabled GPU is detected
    print("GPU:", torch.cuda.get_device_name(0))  # Prints the name of the first detected GPU
    # Prints the total VRAM (Video RAM) of the first GPU in gigabytes, formatted to two decimal places
    print("VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [ ]:
!nvidia-smi

Mon Nov  9 06:07:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Overall Workflow:
This notebook demonstrates a typical workflow for **fine-tuning a pre-trained Large Language Model (LLM)**, specifically Qwen2.5-3B-Instruct, using **Low-Rank Adaptation (LoRA)** for efficiency. It leverages the `unsloth` library to accelerate the fine-tuning process. The general steps involve setting up the environment, loading and configuring the model for LoRA, preparing a small dataset, training the model, and finally saving the fine-tuned adapter.

### Cell-by-cell Explanation:

*   **Cell 1 - Environment Check (PyTorch and CUDA)**:
    *   **What's happening**: This cell imports the `torch` library and checks the PyTorch version, whether CUDA (NVIDIA GPU acceleration) is available, and if so, it prints the name of the GPU and its total VRAM. This is a crucial step to ensure the environment is correctly set up for GPU-accelerated deep learning tasks.
    *   **Purpose**: To confirm that PyTorch is installed and that a compatible GPU is detected and ready for use, which is essential for training large models efficiently.

In [ ]:
!pip install -U unsloth  # Installs or upgrades the unsloth library, a tool for faster LLM fine-tuning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.4/25.4 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119

*   **Cell 2 - Install `unsloth` Library**:
    *   **What's happening**: This cell uses the `pip` package manager to install or upgrade the `unsloth` library. `unsloth` is a library designed to make fine-tuning large language models faster and more memory-efficient.
    *   **Purpose**: To install the necessary tools that will optimize the LoRA fine-tuning process.

In [ ]:
!pip install -U unsloth  # Installs or upgrades the unsloth library (redundant if already installed/upgraded)

*   **Cell 3 - Verify `unsloth` Installation (Redundant)**:
    *   **What's happening**: This cell attempts to install/upgrade `unsloth` again. Since it was just installed, the output indicates that all requirements are already satisfied.
    *   **Purpose**: While redundant in this sequence, it serves to confirm `unsloth` is correctly installed, although the previous cell already achieved this.

In [ ]:

import torch  # Imports the PyTorch library
import unsloth   # Imports the unsloth library for optimized LLM operations
import transformers  # Imports the Hugging Face Transformers library for models and tokenizers
import trl  # Imports the TRL library for Transformer Reinforcement Learning (fine-tuning)
import peft  # Imports the PEFT library for Parameter-Efficient Fine-Tuning

print("CUDA available :", torch.cuda.is_available())  # Checks and prints if CUDA is available
print("GPU            :", torch.cuda.get_device_name(0))  # Prints the name of the active GPU
# Prints the total VRAM of the active GPU in gigabytes
print("VRAM           :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
print("Unsloth        :", unsloth.__version__)  # Prints the unsloth library version
print("Transformers   :", transformers.__version__)  # Prints the Transformers library version
print("TRL            :", trl.__version__)  # Prints the TRL library version
print("PEFT           :", peft.__version__)  # Prints the PEFT library version

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

*   **Cell 4 - Import Libraries and Display Versions**:
    *   **What's happening**: This cell imports key libraries required for the fine-tuning process: `torch`, `unsloth`, `transformers` (for working with pre-trained models), `trl` (Transformer Reinforcement Learning, often used for fine-tuning), and `peft` (Parameter-Efficient Fine-tuning). It then prints the CUDA status, GPU information, and the versions of these imported libraries.
    *   **Purpose**: To make these libraries available for use in subsequent cells and to verify that the expected versions are loaded, which can be important for compatibility.

In [ ]:
from unsloth import FastLanguageModel  # Imports FastLanguageModel from unsloth for optimized model loading

model_name = "Qwen/Qwen2.5-3B-Instruct"  # Defines the name of the pre-trained model to load

max_seq_length = 2048  # Sets the maximum sequence length for the model's input

# Loads the pre-trained model and tokenizer, enabling 4-bit quantization for memory efficiency
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,  # Specifies the model to load
    max_seq_length=max_seq_length,  # Sets the maximum sequence length
    load_in_4bit=True,  # Enables 4-bit quantization for memory reduction
)

print("Model loaded successfully!")  # Confirms successful model loading

*   **Cell 5 - Load Pre-trained Model and Tokenizer**:
    *   **What's happening**: This cell uses `FastLanguageModel.from_pretrained` from `unsloth` to load the `Qwen/Qwen2.5-3B-Instruct` model and its corresponding tokenizer. Key parameters include `max_seq_length` (the maximum sequence length the model can handle) and `load_in_4bit=True` (to load the model in 4-bit quantization, significantly reducing memory usage).
    *   **Purpose**: To load the base large language model that will be fine-tuned. The 4-bit loading is essential for fitting such a large model into limited GPU memory.

In [ ]:
model = FastLanguageModel.get_peft_model(  # Configures the loaded model for PEFT (LoRA)
    model,  # The base model to apply PEFT to

    r=16,  # LoRA rank: determines the size of the low-rank matrices

    target_modules=[  # Specifies which model layers to apply LoRA to
        "q_proj",  # Query projection layer
        "k_proj",  # Key projection layer
        "v_proj",  # Value projection layer
        "o_proj",  # Output projection layer
        "gate_proj",  # Gate projection layer (for gating mechanisms)
        "up_proj",  # Up-projection layer (in MLPs)
        "down_proj",  # Down-projection layer (in MLPs)
    ],

    lora_alpha=16,  # LoRA alpha: scaling factor for LoRA updates
    lora_dropout=0,  # LoRA dropout: dropout rate for LoRA layers
    bias="none",  # Specifies if bias terms are trained (here, none are trained)

    use_gradient_checkpointing="unsloth",  # Enables gradient checkpointing for memory optimization

    random_state=3407,  # Sets a random seed for reproducibility
)

print("LoRA adapters added!")  # Confirms that LoRA adapters have been added to the model

*   **Cell 6 - Configure Model for LoRA Fine-tuning**:
    *   **What's happening**: This cell applies `FastLanguageModel.get_peft_model` to the loaded model. This function configures the model for LoRA (Low-Rank Adaptation) by adding small, trainable adapter layers. Parameters like `r` (LoRA rank), `target_modules` (which parts of the model to apply LoRA to), `lora_alpha`, `lora_dropout`, and `bias` are set. `use_gradient_checkpointing='unsloth'` is also enabled for further memory optimization.
    *   **Purpose**: To prepare the pre-trained model for efficient fine-tuning. Instead of training all billions of parameters, only the much smaller LoRA adapter layers are updated, saving computational resources and time.

In [ ]:
model.print_trainable_parameters()  # Prints a summary of the model's trainable parameters (LoRA adapters)

*   **Cell 7 - Print Trainable Parameters**:
    *   **What's happening**: This cell calls `model.print_trainable_parameters()` to display a summary of the model's parameters.
    *   **Purpose**: To show how many parameters are actually trainable after applying LoRA. You'll notice a very small percentage of the total parameters are trainable, highlighting the efficiency of LoRA.

In [ ]:
from transformers import TextStreamer  # Imports TextStreamer for streaming text generation (though not used directly here)

messages = [  # Defines a list of messages representing a conversational turn
    {  # A single message object
        "role": "user",  # Role of the speaker
        "content": "What is a firewall?"  # The user's query
    }
]

# Applies the model's chat template to the messages, tokenizes them, and prepares for generation
inputs = tokenizer.apply_chat_template(
    messages,  # The list of messages
    tokenize=True,  # Tokenize the input text
    add_generation_prompt=True,  # Adds a prompt to start generation after user input
    return_tensors="pt",  # Returns PyTorch tensors
).to("cuda")  # Moves the tensors to the GPU

# Generates text based on the input messages
outputs = model.generate(
    input_ids=inputs,  # The tokenized input IDs
    max_new_tokens=100,  # Maximum number of new tokens to generate
    temperature=0.7,  # Sampling temperature for controlling randomness in generation
    do_sample=True,  # Enables sampling (vs. greedy decoding)
)

# Decodes the generated token IDs back into human-readable text and prints it
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

*   **Cell 8 - Pre-Fine-tuning Inference Demonstration**:
    *   **What's happening**: This cell demonstrates how to use the model for text generation *before* any fine-tuning. It prepares a chat message using the tokenizer's chat template and then calls `model.generate` to produce an output based on a user query.
    *   **Purpose**: To show the baseline performance and behavior of the pre-trained model before it learns from the new dataset. This helps in comparing performance after fine-tuning.

In [ ]:
from datasets import Dataset  # Imports the Dataset class from the datasets library

# Creates a Dataset object from a list of conversational examples
dataset = Dataset.from_list([
    {  # First conversational example
        "messages": [  # List of messages in the conversation
            {  # User message
                "role": "user",  # Role: user
                "content": "What is a firewall?"  # Content of the user message
            },
            {  # Assistant message (response)
                "role": "assistant",  # Role: assistant
                "content": "A firewall is a security mechanism that monitors and controls network traffic based on predefined rules."  # Content of the assistant's response
            }
        ]
    },
    {  # Second conversational example
        "messages": [  # List of messages
            {  # User message
                "role": "user",  # Role: user
                "content": "What is phishing?"  # Content of the user message
            },
            {  # Assistant message (response)
                "role": "assistant",  # Role: assistant
                "content": "Phishing is a social engineering attack in which an attacker attempts to trick a victim into revealing sensitive information."  # Content of the assistant's response
            }
        ]
    }
])

print(dataset)  # Prints a summary of the created dataset
print("\nFirst example:")  # Prints a header for the first example
print(dataset[0])  # Prints the first example from the dataset

*   **Cell 9 - Create Training Dataset**:
    *   **What's happening**: This cell creates a small `Dataset` object from a Python list of dictionaries. Each dictionary represents a conversational exchange, with `role` (user or assistant) and `content`. This dataset will be used to fine-tune the model.
    *   **Purpose**: To prepare the data in a structured format suitable for the `trl` library's `SFTTrainer` for supervised fine-tuning.

In [ ]:
!nvidia-smi  # Executes the `nvidia-smi` command to display NVIDIA GPU status and usage

*   **Cell 10 - Check GPU Status (nvidia-smi)**:
    *   **What's happening**: This cell runs the `nvidia-smi` command, which is a utility to monitor NVIDIA GPU usage. It displays information like GPU memory usage, temperature, and running processes on the GPU.
    *   **Purpose**: To get a real-time snapshot of the GPU's status, especially memory usage, before and during training to ensure there's enough available memory and to monitor resource consumption.

In [ ]:
from trl import SFTTrainer, SFTConfig  # Imports SFTTrainer and SFTConfig from the TRL library

# Defines the training arguments and configuration using SFTConfig
training_args = SFTConfig(
    output_dir="./qwen2.5-3b-qlora",  # Directory to save model checkpoints and logs

    # Training parameters
    num_train_epochs=3,  # Number of training epochs
    per_device_train_batch_size=1,  # Batch size per GPU
    gradient_accumulation_steps=4,  # Accumulate gradients over 4 steps before updating weights

    # Learning rate schedule
    learning_rate=2e-4,  # Initial learning rate
    warmup_ratio=0.05,  # Proportion of steps for linear warmup of learning rate

    # Sequence length
    max_length=2048,  # Maximum sequence length for training inputs

    # Memory optimization
    gradient_checkpointing=True,  # Enables gradient checkpointing to save memory

    # Optimizer
    optim="adamw_8bit",  # Specifies the 8-bit AdamW optimizer

    # Logging
    logging_steps=1,  # Log metrics every 1 step
    report_to="none",  # Disable reporting to external platforms (e.g., Weights & Biases)

    # Saving strategy
    save_strategy="epoch",  # Save model checkpoints at the end of each epoch

    # Reproducibility
    seed=3407,  # Sets a random seed for reproducibility
)

print("Training configuration created successfully!")  # Confirms successful configuration creation

*   **Cell 11 - Define Training Configuration (`SFTConfig`)**:
    *   **What's happening**: This cell defines the training arguments using `SFTConfig` from `trl`. These arguments include `output_dir` (where to save results), `num_train_epochs`, `per_device_train_batch_size`, `gradient_accumulation_steps` (to simulate larger batch sizes), `learning_rate`, `warmup_ratio`, `max_length`, `gradient_checkpointing` (for memory optimization), `optim` (optimizer type), `logging_steps`, `save_strategy`, and `seed` (for reproducibility).
    *   **Purpose**: To configure all the hyperparameters and settings for the supervised fine-tuning process.

In [ ]:
from datasets import Dataset  # Imports the Dataset class from the datasets library

# Creates a Dataset object from a list of conversational examples (duplicate creation)
dataset = Dataset.from_list([
    {  # First conversational example
        "messages": [  # List of messages
            {  # User message
                "role": "user",  # Role: user
                "content": "What is a firewall?"  # Content
            },
            {  # Assistant message
                "role": "assistant",  # Role: assistant
                "content": "A firewall is a security mechanism that monitors and controls network traffic based on predefined rules."  # Content
            }
        ]
    },
    {  # Second conversational example
        "messages": [  # List of messages
            {  # User message
                "role": "user",  # Role: user
                "content": "What is phishing?"  # Content
            },
            {  # Assistant message
                "role": "assistant",  # Role: assistant
                "content": "Phishing is a social engineering attack in which an attacker attempts to trick a victim into revealing sensitive information."  # Content
            }
        ]
    }
])

*   **Cell 12 - Create Training Dataset (Duplicate)**:
    *   **What's happening**: This cell is a duplicate of the previous dataset creation cell, creating the same `Dataset` object again. This indicates `dataset` is assigned to a variable of the same name again.
    *   **Purpose**: This seems to be a redundant execution. It likely recreates the training dataset. (Note: The next cell refers to `train_dataset`, implying there might be an implicit or explicit assignment `train_dataset = dataset` somewhere not shown in the latest notebook state, or a direct use of `dataset` as `train_dataset`.)

In [ ]:
print(type(train_dataset[0]["text"]))  # Prints the data type of the 'text' field in the first training example
print(train_dataset[0]["text"])  # Prints the formatted text content of the first training example

*   **Cell 13 - Inspect Formatted Training Example**:
    *   **What's happening**: This cell prints the type and content of the first element (`[0]`) from `train_dataset` (presumably the `dataset` created earlier, now referred to as `train_dataset`). This output shows how the raw conversational data is formatted into a single string with special tokens for the model's chat template (`<|im_start|>`, `<|im_end|>`, `system`, `user`, `assistant`).
    *   **Purpose**: To verify that the input data for the trainer is correctly formatted according to the model's chat template, which is crucial for the model to understand the conversation turns.

In [ ]:
from trl import SFTTrainer  # Imports the SFTTrainer class from the TRL library

# Initializes the SFTTrainer with the configured model, tokenizer, dataset, and training arguments
trainer = SFTTrainer(
    model=model,  # The LoRA-configured model to be trained
    processing_class=tokenizer,  # The tokenizer used for processing text
    train_dataset=train_dataset,  # The dataset used for training
    dataset_text_field="text",  # Specifies the field in the dataset that contains the text
    args=training_args,  # The training configuration arguments
)

print("SFTTrainer created successfully!")  # Confirms successful trainer initialization

*   **Cell 14 - Initialize `SFTTrainer`**:
    *   **What's happening**: This cell initializes the `SFTTrainer` from the `trl` library. It passes the LoRA-configured `model`, the `tokenizer`, the `train_dataset`, the `dataset_text_field` (which is 'text' as seen in the previous cell's output), and the `training_args` defined earlier.
    *   **Purpose**: To set up the training pipeline. The `SFTTrainer` handles the training loop, data processing, and optimization based on the provided configurations.

ACTUAL TRAIN

*   **Cell 15 - Markdown Cell (ACTUAL TRAIN)**:
    *   **What's happening**: This is a markdown cell with the text "ACTUAL TRAIN".
    *   **Purpose**: To serve as a clear heading or marker in the notebook, indicating that the following cells are where the main training process will occur.

In [ ]:
import torch  # Imports the PyTorch library

print("GPU:", torch.cuda.get_device_name(0))  # Prints the name of the active GPU
# Prints the currently allocated GPU VRAM in gigabytes
print("VRAM allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
# Prints the currently reserved GPU VRAM in gigabytes
print("VRAM reserved:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")

*   **Cell 16 - Check VRAM Allocation Before Training**:
    *   **What's happening**: This cell imports `torch` and prints the currently allocated and reserved GPU VRAM. This is a more detailed check than the initial `get_device_properties` call.
    *   **Purpose**: To monitor the memory state of the GPU right before training begins, which is useful for debugging out-of-memory errors or optimizing batch sizes.

In [ ]:
train_result = trainer.train()  # Starts the fine-tuning process using the SFTTrainer

print("Training completed successfully!")  # Confirms that training has finished

*   **Cell 17 - Execute Training**:
    *   **What's happening**: This cell initiates the fine-tuning process by calling `trainer.train()`. This command starts the training loop as configured by `SFTTrainer` and `training_args`. The output shows warnings about tokenizers and details about the training process (number of examples, epochs, steps, batch size, trainable parameters).
    *   **Purpose**: This is the core step where the model learns from the provided dataset by updating its LoRA adapter weights.

In [ ]:
output_dir = "./qwen2.5-3b-cyber-lora"  # Defines the directory where the fine-tuned adapter will be saved

model.save_pretrained(output_dir)  # Saves the fine-tuned LoRA adapter weights to the specified directory
tokenizer.save_pretrained(output_dir)  # Saves the tokenizer configuration to the specified directory

print(f"LoRA adapter saved to: {output_dir}")  # Prints the path where the adapter was saved

*   **Cell 18 - Save Fine-tuned LoRA Adapter**:
    *   **What's happening**: After training is complete, this cell saves the fine-tuned LoRA adapter and the tokenizer to a specified directory (`./qwen2.5-3b-cyber-lora`).
    *   **Purpose**: To persist the trained model weights and tokenizer configuration so that the fine-tuned model can be reloaded and used later for inference or further training without needing to retrain from scratch.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="./qwen2.5-3b-cyber-lora",
    max_seq_length=2048,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print("Model and tokenizer loaded successfully!")
print("Tokenizer:", type(tokenizer))

In [ ]:
import torch

prompt = "What is a firewall?"

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=100,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )

response = tokenizer.decode(
    outputs[0][inputs.shape[1]:],
    skip_special_tokens=True
)

print("Question:", prompt)
print("\nFine-tuned model:")
print(response)